<a href="https://colab.research.google.com/github/vipinvicky127-svg/ds_Vipin_Kumar/blob/main/Zopper_Assignment_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- BLOCK 1: IMPORT LIBRARIES & LOAD EXCEL ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 1. Check if file exists
file_name = '/project_data.xls'


if os.path.exists(file_name):
    print(f"✅ Success! Found '{file_name}'. Loading data...")

    # 2. Read the Excel file
    # We use read_excel instead of read_csv because your file is .xls
    df = pd.read_excel(file_name)

    # Show the first 5 rows to ensure it looks right
    print("Here is a preview of your data:")
    display(df.head())

else:
    print(f"❌ Error: Could not find '{file_name}'.")
    print("Please make sure the file name in the folder icon matches exactly.")

In [ ]:
# --- BLOCK 2: CLEANING THE DATA ---

# Define the columns that represent months
month_cols = ['Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# Function to fix messy numbers
def clean_currency(x):
    if isinstance(x, str):
        # If it's text like "23%", remove % and convert to number
        return float(x.replace('%', ''))
    elif isinstance(x, (int, float)):
        # If Excel saved it as 0.23, convert to 23.0
        # If it's already 23.0, keep it as is.
        if x <= 1:
            return x * 100
        return x
    return 0

# Apply the cleaning to all month columns
for col in month_cols:
    df[col] = df[col].apply(clean_currency)

print("✅ Data Cleaned! All percentages are now numbers.")
print(df[month_cols].head()) # Show just the numbers to verify

In [ ]:
# --- BLOCK 3: ANALYSIS & CLUSTERING ---
#This block calculates the Average, Volatility (Risk), and creates the Clusters (grouping similar stores).
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Create Statistics
df['Average_Perf'] = df[month_cols].mean(axis=1) # Average
df['Volatility'] = df[month_cols].std(axis=1)    # Risk (Standard Deviation)
df['Momentum'] = df['Dec'] - df['Aug']           # Growth (Dec vs Aug)

# 2. Machine Learning (Clustering)
# We group stores based on their stats
scaler = StandardScaler()
features = df[['Average_Perf', 'Volatility', 'Momentum']]
scaled_features = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=4, random_state=42)
df['Cluster'] = kmeans.fit_predict(scaled_features)

# Name the clusters automatically
def name_clusters(df):
    summary = df.groupby('Cluster')['Average_Perf'].mean().sort_values(ascending=False)
    rank = summary.index.tolist()
    mapping = {
        rank[0]: 'High Performers',
        rank[1]: 'Growth/Volatile',
        rank[2]: 'Consistent Average',
        rank[3]: 'Low/Inactive'
    }
    return df['Cluster'].map(mapping)

df['Segment'] = name_clusters(df)

print("✅ Analysis Complete. Created Clusters.")
print("\n--- TOP 3 GROWING STORES ---")
print(df.sort_values('Momentum', ascending=False)[['Store_Name', 'Momentum']].head(3))

print("\n--- TOP 3 CRASHING STORES ---")
print(df.sort_values('Momentum', ascending=True)[['Store_Name', 'Momentum']].head(3))

In [ ]:
# --- BLOCK 4: VISUALIZATION DASHBOARD ---
sns.set_theme(style="whitegrid", palette="muted")
plt.figure(figsize=(18, 12))

# 1. Risk vs Reward Scatter Plot
plt.subplot(2, 2, 1)
sns.scatterplot(x='Volatility', y='Average_Perf', hue='Segment', data=df, s=100, palette='deep')
plt.title('Risk vs Reward (Where do stores sit?)', fontsize=14, fontweight='bold')
plt.xlabel('Risk (Volatility)')
plt.ylabel('Average Performance %')

# 2. Regional Boxplot
plt.subplot(2, 2, 2)
# Sort regions by performance
order = df.groupby('Branch')['Average_Perf'].median().sort_values(ascending=False).index
sns.boxplot(x='Average_Perf', y='Branch', data=df, order=order, palette='viridis')
plt.title('Which Region is Best?', fontsize=14, fontweight='bold')

# 3. Heatmap of Pune Region
plt.subplot(2, 2, 3)
pune_data = df[df['Branch'] == 'Pune'].set_index('Store_Name')[month_cols]
sns.heatmap(pune_data, cmap='RdYlGn', annot=True, fmt='.0f', cbar=False)
plt.title('Deep Dive: Pune Region Heatmap', fontsize=14, fontweight='bold')

# 4. Trend Lines (Best vs Worst)
plt.subplot(2, 2, 4)
# Pick top store and a crashing store to compare
targets = df.sort_values('Momentum', ascending=False).iloc[[0, -1]]
target_names = targets['Store_Name'].tolist()
subset = df[df['Store_Name'].isin(target_names)]
melted = subset.melt(id_vars='Store_Name', value_vars=month_cols, var_name='Month', value_name='Score')

sns.lineplot(x='Month', y='Score', hue='Store_Name', data=melted, marker='o', linewidth=3)
plt.title('Trend: The Best Growth vs The Worst Crash', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# --- BLOCK 5: SETUP INTERACTIVE PLOTTING ---
import plotly.express as px
import plotly.graph_objects as go

print("✅ Plotly loaded! Ready to create interactive charts.")

In [ ]:
# --- CHART 1: RISK VS REWARD SCATTER ---

fig1 = px.scatter(
    df,
    x='Volatility',
    y='Average_Perf',
    color='Segment',                 # Color dots by their Cluster
    hover_name='Store_Name',         # Show Store Name when hovering
    hover_data=['Branch', 'Momentum'], # Show extra info
    size='Average_Perf',             # Bigger dots = Higher Performance
    title='<b>Strategic Map: Risk vs Reward</b><br>(Hover over dots to see Store Names)',
    template='plotly_white',
    height=600
)

# Add a vertical line to show where "High Risk" starts
fig1.add_vline(x=15, line_width=1, line_dash="dash", line_color="red")
fig1.add_annotation(x=16, y=90, text="High Risk Zone >>", showarrow=False)

fig1.show()

In [ ]:
# --- CHART 2: REGIONAL PERFORMANCE ---

# Sort data so the best region is on top
df_sorted = df.sort_values('Average_Perf', ascending=False)

fig2 = px.box(
    df_sorted,
    x='Branch',
    y='Average_Perf',
    color='Branch',
    points="all", # Show the actual store dots next to the box
    hover_name='Store_Name',
    title='<b>Regional Performance Comparison</b><br>(Which region is winning?)',
    template='plotly_white',
    height=500
)

fig2.update_layout(showlegend=False) # Hide legend because x-axis already shows names
fig2.show()

In [ ]:
# --- CHART 3: PUNE HEATMAP ---

# 1. Filter for Pune only
pune_df = df[df['Branch'] == 'Pune'].set_index('Store_Name')[month_cols]

# 2. Create Heatmap
fig3 = px.imshow(
    pune_df,
    labels=dict(x="Month", y="Store", color="Performance %"),
    x=month_cols,
    y=pune_df.index,
    color_continuous_scale='Viridis', # Color scheme
    aspect="auto",
    title='<b>Deep Dive: Pune Region Volatility</b><br>(Yellow = High Perf, Purple = Low Perf)',
    height=700
)

fig3.show()

In [ ]:
# --- CHART 4: TREND LINES ---

# 1. Get Top 5 and Bottom 5 stores based on Momentum (Growth)
top_5 = df.nlargest(5, 'Momentum')
bottom_5 = df.nsmallest(5, 'Momentum')
combined_movers = pd.concat([top_5, bottom_5])

# 2. Melt data (Format it for line charts)
melted_df = combined_movers.melt(
    id_vars=['Store_Name', 'Segment'],
    value_vars=month_cols,
    var_name='Month',
    value_name='Performance'
)

# 3. Draw the Lines
fig4 = px.line(
    melted_df,
    x='Month',
    y='Performance',
    color='Store_Name',
    markers=True, # Add dots on the lines
    title='<b>Trend Analysis: Top 5 Growers vs. Top 5 Crashers</b><br>(Click legend to hide/show lines)',
    template='plotly_white',
    height=600
)

# Highlight the 100% mark
fig4.add_hline(y=100, line_dash="dot", annotation_text="Max Possible (100%)", annotation_position="top left")

fig4.show()

In [ ]:
# --- MODULE 5: RELIABILITY ANALYSIS ---

# 1. Calculate Coefficient of Variation (CV)
# Formula: Volatility / Average. Lower is better.
df['Reliability_Score'] = df['Volatility'] / df['Average_Perf']

# Filter out zeros to avoid errors
active_stores = df[df['Average_Perf'] > 5].copy()

# Sort: Lowest score is most reliable
most_reliable = active_stores.sort_values('Reliability_Score').head(10)

# Visualize
fig5 = px.bar(
    most_reliable,
    x='Reliability_Score',
    y='Store_Name',
    color='Average_Perf',
    orientation='h',
    title='<b>Top 10 Most Consistent Stores (The "Anchors")</b><br>(Short bars = Very Stable. Color = Performance)',
    labels={'Reliability_Score': 'Instability Score (Lower is Better)'},
    template='plotly_white',
    color_continuous_scale='Teal'
)

fig5.update_layout(yaxis=dict(autorange="reversed")) # Top store at top
fig5.show()

print("STRATEGY TIP: These stores are your 'Anchors'. Use them to pilot new products because their baseline is stable.")

In [ ]:
# --- MODULE 6: REGIONAL HEALTH CHECK (ZOMBIES) ---

# 1. Tag stores as Zombie or Active
# We assume anything averaging under 5% is effectively dead
df['Health_Status'] = np.where(df['Average_Perf'] < 5, 'Zombie (Inactive)', 'Healthy (Active)')

# 2. Count them by Region
regional_health = df.groupby(['Branch', 'Health_Status']).size().reset_index(name='Count')

# 3. Visualize
fig6 = px.bar(
    regional_health,
    x='Branch',
    y='Count',
    color='Health_Status',
    title='<b>Regional Health: Where are the "Zombie" Stores?</b><br>(Red bars = Dead Capital/Potential Closures)',
    color_discrete_map={'Zombie (Inactive)': '#EF553B', 'Healthy (Active)': '#00CC96'},
    template='plotly_white',
    barmode='stack'
)

fig6.show()

print("STRATEGY TIP: Look at 'Thane' and 'Mumbai'. The red bars represent rent paid for zero return. Initiate closures or re-staffing immediately.")

In [ ]:
# --- MODULE 7: PREDICTIVE FORECAST FOR JANUARY ---

# 1. Calculate Weighted Forecast
# We give 50% weight to Dec, 30% to Nov, 20% to Oct
df['Jan_Forecast'] = (df['Dec'] * 0.5) + (df['Nov'] * 0.3) + (df['Oct'] * 0.2)

# 2. Filter Top Opportunities (Stores likely to cross 50% next month)
opportunities = df[df['Jan_Forecast'] > 40].sort_values('Jan_Forecast', ascending=False).head(15)

# 3. Visualize
fig7 = px.scatter(
    opportunities,
    x='Jan_Forecast',
    y='Store_Name',
    size='Jan_Forecast',
    color='Branch',
    title='<b>January Forecast: Top 15 Stores Predicted to Exceed Targets</b><br>(Allocate extra inventory to these locations)',
    labels={'Jan_Forecast': 'Predicted Performance %'},
    template='plotly_white',
    height=600
)

# Add a target line
fig7.add_vline(x=50, line_dash="dash", line_color="green", annotation_text="Target Goal")

fig7.show()

print("STRATEGY TIP: Ensure these 15 stores have full stock for January. They are trending to hit the highest numbers.")

In [ ]:
# --- MODULE 7 (FIXED): AI FORECASTING & INTERACTIVE PREDICTOR ---
from sklearn.linear_model import LinearRegression
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.express as px
import plotly.graph_objects as go

# ---------------------------------------------------------
# STEP 1: TRAIN THE AI MODEL
# ---------------------------------------------------------
# X (Features) = Aug, Sep, Oct, Nov
# y (Target)   = Dec
X_train_data = df[['Aug', 'Sep', 'Oct', 'Nov']]
y_train_data = df['Dec']

model = LinearRegression()
model.fit(X_train_data, y_train_data)

# Predict January (using Sep, Oct, Nov, Dec as input)
X_future = df[['Sep', 'Oct', 'Nov', 'Dec']]
X_future.columns = ['Aug', 'Sep', 'Oct', 'Nov'] # Rename to match training columns
df['AI_Jan_Prediction'] = model.predict(X_future)
df['AI_Jan_Prediction'] = df['AI_Jan_Prediction'].clip(lower=0, upper=100)

print("✅ AI Model Trained & January Predictions Generated.")

# ---------------------------------------------------------
# STEP 2: THE FIXED PREDICTION FUNCTION
# ---------------------------------------------------------
def predict_store(store_name_input):
    # --- THE FIX IS HERE: regex=False ---
    # This tells Python to treat '(' and ')' as normal text
    store_match = df[df['Store_Name'].str.contains(store_name_input, case=False, na=False, regex=False)]

    if store_match.empty:
        # Fallback: If exact match fails, try generic search (useful for typing 'Pune')
        # We turn regex=True back on for generic words, but escape special chars if needed
        try:
            store_match = df[df['Store_Name'].str.contains(store_name_input, case=False, na=False)]
        except:
            print(f"❌ Could not find store: {store_name_input}")
            return

    if store_match.empty:
        print(f"❌ Store '{store_name_input}' not found.")
        return

    # Pick the first result
    store_row = store_match.iloc[0]
    name = store_row['Store_Name']

    # Get Data
    history = [store_row['Aug'], store_row['Sep'], store_row['Oct'], store_row['Nov'], store_row['Dec']]
    prediction = store_row['AI_Jan_Prediction']
    months = ['Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Jan (Pred)']
    values = history + [prediction]

    # VISUALIZE
    print(f"\n📊 PREDICTION FOR: {name}")
    print(f"   - Current Trend: {'UP 📈' if prediction > history[-1] else 'DOWN 📉'}")
    print(f"   - AI Forecast for Jan: {prediction:.1f}%")

    fig = px.line(
        x=months,
        y=values,
        markers=True,
        title=f'<b>AI Forecast: {name}</b>',
        template='plotly_white'
    )

    # Add Red Dotted Line for the Prediction
    fig.add_trace(go.Scatter(
        x=['Dec', 'Jan (Pred)'],
        y=[history[-1], prediction],
        mode='lines+markers',
        line=dict(color='red', width=3, dash='dot'),
        name='AI Forecast'
    ))

    fig.update_layout(showlegend=False, yaxis_range=[0, 110])
    fig.show()

# ---------------------------------------------------------
# STEP 3: DROPDOWN MENU
# ---------------------------------------------------------
all_stores = sorted(df['Store_Name'].unique().tolist())

print("\n👇 SELECT A STORE TO PREDICT:")

dropdown = widgets.Dropdown(
    options=all_stores,
    description='Store:',
    disabled=False,
)

def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        clear_output(wait=True)
        display(dropdown)
        predict_store(change['new'])

dropdown.observe(on_change)
display(dropdown)

# Initialize with the first store
predict_store(all_stores[0])